# C9 — Famille A sur ESC-50 : CNN et CRNN sur le log-Mel (Brique 4, P3, P4, P6, P8)

Ce notebook entraîne les deux réseaux de la famille A sur les 2 000 extraits d'ESC-50, avec la
validation croisée sur les 5 folds officiels (P3) :

1. le log-Mel de chaque extrait est calculé **une seule fois** (R5, R6) par
   `echolens.core.features`, puis gardé en sortie du notebook ;
2. à chaque tour k, réglage du taux d'apprentissage et du nombre d'époques (3 folds pour
   apprendre, fold k + 1 pour l'early stopping), puis 3 entraînements finaux (graines 42, 43,
   44) sur les 4 folds hors test et une seule évaluation sur le fold k ;
3. la standardisation par bande (R4) est recalculée sur l'entraînement de chaque étape ; les
   augmentations (P6) ne touchent que l'entraînement.

Tout le calcul vient du dépôt (`echolens.training.family_a*`) ; le notebook enchaîne les étapes,
vérifie l'absence de fuite et enregistre les résultats. Aucune prédiction, aucun poids et aucun
log-Mel ne va sur GitHub : seuls les JSON de résultats et les figures y vont.

## a. Démarrage

On clone le dépôt (branche `BRANCH`) et on l'installe en mode éditable (I9), **sans** l'extra
`[ml]` : torch est déjà installé sur Kaggle. L'installation éditable n'est visible qu'après un
redémarrage du noyau : on ajoute donc `src/` au `sys.path`.

`SMOKE_TEST = True` fait un essai à blanc de tout le notebook en quelques minutes (2 époques au
plus) : à utiliser seulement pour vérifier le notebook, jamais pour des résultats.

In [ ]:
import sys

BRANCH = "main"
SMOKE_TEST = False
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. ffmpeg et GPU

`load_audio` décode avec ffmpeg (D1). L'entraînement se fait sur le GPU (T4), en float32, sans
précision mixte : sans GPU, on arrête plutôt que d'entraîner pendant des heures sur CPU.

In [ ]:
import shutil
import time

import torch

if shutil.which("ffmpeg") is None:
    !apt-get -qq update && apt-get -qq install -y ffmpeg
print("ffmpeg :", shutil.which("ffmpeg"))

if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU : activer l'accélérateur GPU T4 du notebook.")
device = torch.device("cuda", 0)
GPU_NAME = torch.cuda.get_device_name(device)
print("GPU :", GPU_NAME)
print("torch", torch.__version__)
NOTEBOOK_START = time.time()

## c. Trouver ESC-50

On cherche `meta/esc50.csv` dans les datasets attachés, sinon on télécharge l'archive
officielle depuis GitHub. Le manifeste donne des chemins relatifs à `<datasets_dir>/esc50` :
on crée `/kaggle/working/datasets/esc50`, lien vers la racine trouvée.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
DATASETS_DIR = Path("/kaggle/working/datasets")
DOWNLOAD_DIR = Path("/kaggle/working/downloads")
ESC50_ARCHIVE_URL = "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip"


def find_esc50_root() -> Path:
    found = sorted(INPUT_DIR.glob("**/meta/esc50.csv"))
    if found:
        return found[0].parent.parent
    # Archive officielle (~600 Mo), extraite dans /kaggle/working/downloads.
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = DOWNLOAD_DIR / "esc50.zip"
    urllib.request.urlretrieve(ESC50_ARCHIVE_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DOWNLOAD_DIR)
    archive_path.unlink()
    return DOWNLOAD_DIR / "ESC-50-master"


esc50_root = find_esc50_root()
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
link = DATASETS_DIR / "esc50"
if link.is_symlink():
    link.unlink()
link.symlink_to(esc50_root)
print("esc50 →", esc50_root)

## d. Manifeste versionné et noms des classes

On lit `data/manifests/esc50.csv` (construit par C5) : 2 000 lignes, 400 par fold, chaque audio
présent. Les noms des 50 classes viennent de `meta/esc50.csv` ; on vérifie que `target // 10`
donne bien les 5 catégories (première classe de chaque groupe de 10), comme en C5.

In [ ]:
import csv
from collections import Counter

from echolens.core.data.manifest import audio_path, load_manifest

EXPECTED_CLIPS = 2000
EXPECTED_PER_FOLD = 400
EXPECTED_FIRST_CLASSES = ["dog", "rain", "crying_baby", "door_wood_knock", "helicopter"]

rows = load_manifest(Path(CLONE_DIR) / "data" / "manifests" / "esc50.csv")
missing = [row.path for row in rows if not audio_path(row, DATASETS_DIR).exists()]
if missing:
    raise RuntimeError(f"{len(missing)} audios introuvables, ex. {missing[:3]}")
rows_per_fold = Counter(row.fold for row in rows)
print("Lignes :", len(rows), "· par fold :", dict(sorted(rows_per_fold.items())))
if len(rows) != EXPECTED_CLIPS or set(rows_per_fold.values()) != {EXPECTED_PER_FOLD}:
    raise RuntimeError("Le manifeste d'ESC-50 n'a pas 2 000 lignes réparties en 5 × 400.")

category_of_target = {}
with (esc50_root / "meta" / "esc50.csv").open(encoding="utf-8") as handle:
    for record in csv.DictReader(handle):
        category_of_target[int(record["target"])] = record["category"]
class_names = [category_of_target[target] for target in range(50)]
first_classes = [class_names[group * 10] for group in range(5)]
if first_classes != EXPECTED_FIRST_CLASSES:
    raise RuntimeError("target // 10 ne donne pas les 5 catégories attendues : arrêt.")
print("Classes :", len(class_names))

## e. Log-Mel des 2 000 extraits, calculés une seule fois (R5, R6)

Chaque extrait est chargé en mono 16 kHz (`load_audio`), puis passe par
`echolens.core.features.log_mel_spectrogram` avec `configs/features.yaml` : 64 bandes Mel en dB
absolus, 501 trames pour 5 s. Le tout (2 000 × 64 × 501 en float32, environ 256 Mo) est écrit
dans `esc50_log_mels.npz` et l'entraînement le lit **entièrement en mémoire**. Aucun autre
calcul de features n'a lieu dans ce notebook.

In [ ]:
import numpy as np

from echolens.config import CONFIG_DIR
from echolens.core.features import FeatureConfig
from echolens.training.family_a import compute_esc50_log_mels, load_log_mels, save_log_mels

OUTPUT_DIR = Path("/kaggle/working")
LOG_MELS_PATH = OUTPUT_DIR / "esc50_log_mels.npz"
feature_config = FeatureConfig.from_yaml(CONFIG_DIR / "features.yaml")
# 5 s à 16 kHz = 80 000 échantillons ; trames centrées, pas de 160 → 1 + 80 000 / 160 = 501.
expected_frames = 1 + 5 * feature_config.sample_rate // feature_config.hop_length

start = time.time()
if LOG_MELS_PATH.exists():
    log_mel_set = load_log_mels(LOG_MELS_PATH)
else:
    log_mel_set = compute_esc50_log_mels(rows, DATASETS_DIR, feature_config)
    save_log_mels(log_mel_set, LOG_MELS_PATH)
log_mel_seconds = time.time() - start

expected_shape = (EXPECTED_CLIPS, feature_config.n_mels, expected_frames)
print("Log-Mel :", log_mel_set.log_mels.shape, f"en {log_mel_seconds:.0f} s")
if log_mel_set.log_mels.shape != expected_shape:
    raise RuntimeError(f"Forme {log_mel_set.log_mels.shape} au lieu de {expected_shape}")
if not np.all(np.isfinite(log_mel_set.log_mels)):
    raise RuntimeError("Valeurs NaN ou infinies dans les log-Mel")
if log_mel_set.paths != [row.path for row in rows]:
    raise RuntimeError("L'ordre des log-Mel n'est pas celui du manifeste")
print("Plage (dB) :", float(log_mel_set.log_mels.min()), "à", float(log_mel_set.log_mels.max()))

## f. Configuration et contrôles avant l'entraînement (R4, P3)

Pour chaque tour, on calcule les positions des extraits de chaque partie, puis on vérifie :

- **aucun extrait du fold de test** dans l'entraînement du réglage ni dans l'entraînement final
  (comparaison des chemins de fichiers, pas seulement des numéros de fold) ;
- les **statistiques de standardisation** (moyenne et écart-type des 64 bandes) sont différentes
  d'un tour à l'autre, et entre le réglage (3 folds) et l'entraînement final (4 folds) : elles
  sont bien recalculées à chaque fois sur l'entraînement du moment, jamais sur le test.

In [ ]:
import pandas as pd

from echolens.core.data.splits import esc50_round
from echolens.training.family_a import band_statistics, load_family_a_config
from echolens.training.family_a_esc50 import round_indices

FOLD_COUNT = 5
CLASS_COUNT = len(class_names)
family_a_config = load_family_a_config()
if SMOKE_TEST:
    # Essai à blanc : 2 époques au plus, pour vérifier le notebook de bout en bout.
    smoke_update = {"max_epochs": 2, "patience": 1}
    smoke_training = family_a_config.training.model_copy(update=smoke_update)
    family_a_config = family_a_config.model_copy(update={"training": smoke_training})
print(family_a_config.model_dump())


def check_no_test_clip_in_training(indices, paths) -> None:
    test_paths = {paths[index] for index in indices.test}
    for name in ["tuning_train", "validation", "final_train"]:
        part_paths = {paths[index] for index in getattr(indices, name)}
        shared = test_paths & part_paths
        if shared:
            raise RuntimeError(f"Fuite : {len(shared)} extraits du test dans {name}")


statistics_rows = []
all_statistics = []
for test_fold in range(1, FOLD_COUNT + 1):
    roles = esc50_round(test_fold, FOLD_COUNT)
    indices = round_indices(log_mel_set, roles)
    check_no_test_clip_in_training(indices, log_mel_set.paths)
    tuning_statistics = band_statistics(log_mel_set.log_mels[indices.tuning_train])
    final_statistics = band_statistics(log_mel_set.log_mels[indices.final_train])
    for step, statistics, folds in [
        ("réglage", tuning_statistics, roles.tuning_train_folds),
        ("final", final_statistics, roles.final_train_folds),
    ]:
        all_statistics.append(statistics.mean)
        statistics_rows.append(
            {
                "tour (fold de test)": test_fold,
                "étape": step,
                "folds appris": str(folds),
                "moyenne bande 0 (dB)": round(float(statistics.mean[0]), 3),
                "moyenne bande 32 (dB)": round(float(statistics.mean[32]), 3),
                "écart-type bande 32 (dB)": round(float(statistics.std[32]), 3),
            }
        )
display(pd.DataFrame(statistics_rows))

for first in range(len(all_statistics)):
    for second in range(first + 1, len(all_statistics)):
        if np.allclose(all_statistics[first], all_statistics[second]):
            raise RuntimeError("Deux étapes ont les mêmes statistiques : non recalculées")
print("Aucun extrait de test dans un entraînement ; 10 jeux de statistiques tous différents.")

## g. Entraînement des deux réseaux

Pour chaque architecture, puis chaque fold de test k, `run_esc50_round` :

1. **réglage** : un entraînement par taux d'apprentissage (1e-3 puis 3e-4, graine 42) sur les
   3 folds du réglage, F1 macro sur le fold k + 1 après chaque époque, arrêt après 15 époques
   sans amélioration (150 au plus) ; le meilleur couple (taux, époque) est retenu ;
2. **entraînement final** : avec ce taux et ce nombre d'époques, sans early stopping, sur les
   4 folds hors test, pour les graines 42, 43 et 44 ; une seule évaluation sur le fold k.

Après chaque tour, on écrit ses tableaux (`c9/arrays/<réseau>_fold<k>.npz` : prédictions,
probabilités, poids d'attention du CRNN, statistiques de standardisation) et les poids des
3 modèles finaux (`c9/weights/<réseau>_fold<k>_seed<graine>.pt`, avec leurs statistiques).

In [ ]:
from echolens.training.family_a_esc50 import run_esc50_round
from echolens.training.family_a_report import save_round_arrays

ARCHITECTURES = ["cnn", "crnn"]
ARRAYS_DIR = OUTPUT_DIR / "c9" / "arrays"
WEIGHTS_DIR = OUTPUT_DIR / "c9" / "weights"

rounds_by_architecture = {}
gpu_seconds_by_architecture = {}
for architecture in ARCHITECTURES:
    architecture_start = time.time()
    rounds = []
    for test_fold in range(1, FOLD_COUNT + 1):
        round_start = time.time()
        result = run_esc50_round(
            architecture,
            log_mel_set,
            test_fold,
            FOLD_COUNT,
            CLASS_COUNT,
            family_a_config,
            device,
            WEIGHTS_DIR,
        )
        check_no_test_clip_in_training(result.indices, log_mel_set.paths)
        save_round_arrays(result, architecture, ARRAYS_DIR)
        rounds.append(result)
        chosen = result.tuning.chosen
        seed_texts = [f"{run.seed}: {run.macro_f1:.3f}" for run in result.runs]
        f1_text = ", ".join(seed_texts)
        minutes = (time.time() - round_start) / 60
        print(
            f"{architecture} · fold {test_fold} : lr {chosen.learning_rate:g}, "
            f"{chosen.best_epoch} époques (F1 val {chosen.best_validation_f1:.3f}) · "
            f"F1 test {f1_text} · {minutes:.1f} min"
        )
    rounds_by_architecture[architecture] = rounds
    gpu_seconds_by_architecture[architecture] = time.time() - architecture_start
    print(f"{architecture} : {gpu_seconds_by_architecture[architecture] / 60:.1f} min au total")

## h. Résultats en JSON, un fichier par réseau (P3, P4, P8)

- F1 macro et accuracy : moyenne et écart-type (ddof = 0) sur les 5 tours des scores moyennés
  sur les 3 graines ; détail par graine et écart-type entre graines ;
- sur les 2 000 prédictions réunies des 5 folds de test, **graine 42** : intervalle de confiance
  bootstrap à 95 % du F1 macro, F1 par classe, matrices de confusion normalisées en 50 classes
  et en 5 catégories ;
- par tour : taux d'apprentissage et nombre d'époques retenus, courbes de F1 de validation.

Les prédictions de la graine 42 sont aussi écrites en CSV (`path, fold, target, prediction`),
le format commun qui sert à McNemar contre la référence SVM et la famille B.

In [ ]:
import datetime
import json
import subprocess

import yaml

from echolens.training.esc50_svm import SvmConfig
from echolens.training.evaluation import EvaluationConfig, save_predictions
from echolens.training.family_a_report import (
    cross_validation_scores,
    pooled_scores,
    pooled_test_predictions,
    round_details,
    seed_scores,
)

REFERENCE_SEED = 42
evaluation_config = EvaluationConfig.from_yaml(CONFIG_DIR / "evaluation.yaml")
# Noms français des 5 catégories, les mêmes que pour la référence SVM.
category_names = SvmConfig.from_yaml(CONFIG_DIR / "baseline_svm.yaml").category_names
commit = subprocess.run(
    ["git", "-C", CLONE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()

reports = {}
for architecture, rounds in rounds_by_architecture.items():
    pooled = pooled_test_predictions(rounds, log_mel_set.paths, REFERENCE_SEED)
    save_predictions(pooled, OUTPUT_DIR / f"esc50_{architecture}_test_predictions_seed42.csv")
    report = {
        "experiment": f"esc50_family_a_{architecture}",
        "smoke_test": SMOKE_TEST,
        "date": datetime.datetime.now(datetime.UTC).isoformat(),
        "branch": BRANCH,
        "commit": commit,
        "gpu": GPU_NAME,
        "gpu_seconds": gpu_seconds_by_architecture[architecture],
        "torch_version": torch.__version__,
    }
    report.update(cross_validation_scores(rounds))
    report.update(seed_scores(rounds, family_a_config.training.final_seeds))
    report["reference_seed"] = REFERENCE_SEED
    report["pooled_reference_seed"] = pooled_scores(
        pooled, class_names, category_names, evaluation_config
    )
    report["per_round"] = [round_details(result) for result in rounds]
    report["family_a_config"] = family_a_config.model_dump()
    features_text = (CONFIG_DIR / "features.yaml").read_text(encoding="utf-8")
    report["features_config"] = yaml.safe_load(features_text)
    json_path = OUTPUT_DIR / f"esc50_family_a_{architecture}.json"
    json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    reports[architecture] = report
    print(
        f"{architecture} : F1 macro {report['macro_f1_mean']:.3f} ± {report['macro_f1_std']:.3f}, "
        f"accuracy {report['accuracy_mean']:.3f} ± {report['accuracy_std']:.3f}, "
        f"IC 95 % (graine 42) {report['pooled_reference_seed']['macro_f1_ci']}"
    )

## i. Tableaux : réglage par tour, classes les mieux et les moins bien reconnues

In [ ]:
round_rows = []
for architecture, report in reports.items():
    for details in report["per_round"]:
        seed_f1 = [run["macro_f1"] for run in details["final_runs"]]
        round_rows.append(
            {
                "réseau": architecture,
                "fold de test": details["test_fold"],
                "lr retenu": details["chosen_learning_rate"],
                "époques retenues": details["chosen_epochs"],
                "F1 test (moyenne des graines)": round(float(np.mean(seed_f1)), 4),
            }
        )
display(pd.DataFrame(round_rows))

for architecture, report in reports.items():
    # Classement du F1 par classe, de la mieux à la moins bien reconnue (graine 42).
    ranking = report["pooled_reference_seed"]["per_class_f1"]
    print(f"{architecture}, les mieux reconnues :")
    for entry in ranking[:5]:
        print(f"  {entry['class']} : {entry['f1']:.3f}")
    print(f"{architecture}, les moins bien reconnues :")
    for entry in ranking[-5:]:
        print(f"  {entry['class']} : {entry['f1']:.3f}")

## j. Figures : matrices en 5 catégories et attention du CRNN

Matrices normalisées par ligne sur les 2 000 prédictions de test de la graine 42 (case (i, j) =
part des extraits de la catégorie i prédits dans la catégorie j). Puis, pour quelques extraits
de test bien classés par le CRNN, le log-Mel avec, dessous, les poids d'attention de ses 31 pas
de temps (environ 160 ms chacun) : où le réseau a « regardé » pour décider.

In [ ]:
import matplotlib.pyplot as plt

from echolens.training.family_a_report import run_for_seed

for architecture, report in reports.items():
    matrix = np.array(report["pooled_reference_seed"]["confusion_categories"])
    figure, axes = plt.subplots(figsize=(8, 6))
    image = axes.imshow(matrix, cmap="Blues", vmin=0.0, vmax=1.0)
    axes.set_xticks(range(len(category_names)), category_names, rotation=30, ha="right")
    axes.set_yticks(range(len(category_names)), category_names)
    axes.set_xlabel("Catégorie prédite")
    axes.set_ylabel("Vraie catégorie")
    axes.set_title(f"ESC-50, {architecture.upper()} sur log-Mel (5 folds, graine 42)")
    for i in range(len(category_names)):
        for j in range(len(category_names)):
            color = "white" if matrix[i, j] > 0.5 else "black"
            axes.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", color=color)
    figure.colorbar(image, ax=axes, label="Part de la vraie catégorie")
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / f"esc50_{architecture}_confusion_categories.png", dpi=150)
    plt.show()

EXAMPLE_CLASSES = ["dog", "door_wood_knock", "clock_tick", "siren"]
crnn_first_round = rounds_by_architecture["crnn"][0]
reference_run = run_for_seed(crnn_first_round, REFERENCE_SEED)
figure, axes = plt.subplots(2, len(EXAMPLE_CLASSES), figsize=(16, 5), height_ratios=[3, 1])
for column, class_name in enumerate(EXAMPLE_CLASSES):
    # Premier extrait de test de cette classe, de préférence bien classé.
    target = class_names.index(class_name)
    is_class = crnn_first_round.test_labels == target
    is_correct = reference_run.predictions == target
    candidates = np.flatnonzero(is_class & is_correct)
    if len(candidates) == 0:
        candidates = np.flatnonzero(is_class)
    position = int(candidates[0])
    clip_index = int(crnn_first_round.indices.test[position])
    log_mel = log_mel_set.log_mels[clip_index]
    attention = reference_run.attention[position]
    # Même échelle de couleurs fixe que l'application (U6), en dB absolus.
    axes[0, column].imshow(
        log_mel,
        origin="lower",
        aspect="auto",
        cmap=feature_config.display_colormap,
        vmin=feature_config.display_db_min,
        vmax=feature_config.display_db_max,
    )
    axes[0, column].set_title(f"{class_name} (fold 1)")
    axes[1, column].bar(range(len(attention)), attention)
    axes[1, column].set_xlabel("pas de temps du CRNN")
axes[0, 0].set_ylabel("bande Mel")
axes[1, 0].set_ylabel("attention")
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "esc50_crnn_attention_examples.png", dpi=150)
plt.show()

## k. Bilan du temps et nettoyage

On supprime le lien vers le jeu, l'archive et le clone du dépôt : les sorties gardent les
log-Mel, les tableaux de prédictions, les poids, les JSON, les CSV et les figures.

In [ ]:
total_minutes = (time.time() - NOTEBOOK_START) / 60
print(f"Durée totale du notebook (GPU allumé) : {total_minutes:.0f} min")
for architecture, seconds in gpu_seconds_by_architecture.items():
    print(f"  entraînement {architecture} : {seconds / 60:.0f} min")

shutil.rmtree(DATASETS_DIR, ignore_errors=True)
shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
shutil.rmtree(CLONE_DIR, ignore_errors=True)
print(sorted(path.name for path in OUTPUT_DIR.iterdir()))